# First-Order Feature Analysis: Per Tumor Subregion (WT, TC, ET)

This notebook extends `Hist_Based_Features_Generate_Charts.ipynb` by analyzing first-order
intensity features across **all three tumor subregions** (WT, TC, ET) rather than WT only.

For each feature × MRI modality combination, good vs poor segmentations are compared
with 3 groups: Whole Tumor (WT), Tumor Core (TC), Enhancing Tumor (ET).

Outputs:
- `../../Results/final_figures/firstorder_subregion_<feature>.pdf` — one PDF per feature
- `../../Results/Json_summary/summary_firstorder_subregion.json` — statistics summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import mannwhitneyu
import json
import pickle as pkl
import warnings

warnings.filterwarnings('ignore')


## Helper Functions

In [ ]:
def load_unet_result(path):
    df = pd.read_csv(path, index_col='Unnamed: 0')
    df.index = [idx.split('-seg')[0] for idx in df.index]
    df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard'], axis=1, inplace=True)
    return df

def load_nnunet_result(path):
    with open(path) as f:
        data = json.load(f)
    WT, TC, ET, names = [], [], [], []
    for case in data['metric_per_case']:
        WT.append(case['metrics']['(2, 1, 3)']['Dice'])
        TC.append(case['metrics']['(2, 3)']['Dice'])
        ET.append(case['metrics']['(3,)']['Dice'])
        names.append(case['reference_file'].split('/')[-1].split('.')[0])
    return pd.DataFrame(zip(WT, TC, ET), columns=['WT dice', 'TC dice', 'ET dice'], index=names)

def load_TransBTS_result(path):
    with open(path) as f:
        data = json.load(f)
    WT, TC, ET, names = [], [], [], []
    for case_id, case in data.items():
        WT.append(case['WT'][0])
        TC.append(case['TC'][0])
        ET.append(case['ET'][0])
        names.append(case_id)
    return pd.DataFrame(zip(WT, TC, ET), columns=['WT dice', 'TC dice', 'ET dice'], index=names)

def get_overlaps(unet_df, TransBTS_df, nnunet_df,
                 WT_thresh=0.91, TC_thresh=0.86, ET_thresh=0.85):
    def bad_idx(df):
        return set(df[(df['WT dice'] < WT_thresh) &
                      (df['TC dice'] < TC_thresh) &
                      (df['ET dice'] < ET_thresh)].index)
    return list(bad_idx(unet_df) & bad_idx(nnunet_df))

def load_firstorder(location):
    """Load firstorder pickle from Radiomics/<location>/firstorder.pkl"""
    path = f'../../Results/Analysis_Results/Radiomics/{location}/firstorder.pkl'
    with open(path, 'rb') as f:
        raw = pkl.load(f)
    frames = []
    for feat_name, patient_dict in raw.items():
        feat_df = pd.DataFrame.from_dict(patient_dict, orient='index').astype(float)
        feat_df.columns = [f'{feat_name}_{col}' for col in feat_df.columns]
        frames.append(feat_df)
    return pd.concat(frames, axis=1)

def split_good_bad(df, performance_df, overlaps,
                   WT_thresh=0.91, TC_thresh=0.86, ET_thresh=0.85):
    merged = df.merge(performance_df, left_index=True, right_index=True)
    merged = merged.loc[:, ~merged.columns.duplicated(keep='first')]
    merged = merged.dropna()
    bad = merged[(merged['WT dice'] < WT_thresh) &
                 (merged['TC dice'] < TC_thresh) &
                 (merged['ET dice'] < ET_thresh)]
    bad = bad[bad.index.isin(overlaps)]
    good = merged[(merged['WT dice'] >= WT_thresh) &
                  (merged['TC dice'] >= TC_thresh) &
                  (merged['ET dice'] >= ET_thresh)]
    return good.drop(['WT dice', 'TC dice', 'ET dice'], axis=1), bad.drop(['WT dice', 'TC dice', 'ET dice'], axis=1)

def filter_numeric(df):
    return df.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all')

def cliffs_delta(x, y):
    n_x, n_y = len(x), len(y)
    N_gr = sum(xi > yi for xi in x for yi in y)
    N_ls = sum(xi < yi for xi in x for yi in y)
    return (N_gr - N_ls) / (n_x * n_y)

def categorize_effect_size(delta):
    d = abs(delta)
    if d < 0.147:  return 'No'
    elif d < 0.33: return 'Small'
    elif d < 0.474: return 'Medium'
    else:           return 'Large'

def add_significance_indicator(ax, good, bad, x_pos, alpha=0.05):
    stat, p = mannwhitneyu(good, bad)
    if p < alpha:
        delta = cliffs_delta(good, bad)
        effect = categorize_effect_size(delta)
        ax_top = ax.secondary_xaxis('top')
        ax_top.set_xticks([x_pos + 0.5])
        ax_top.set_xticklabels([f'* {effect}'], rotation=0, ha='center')
        for lbl in ax_top.get_xticklabels():
            lbl.set_color('blue')
            lbl.set_fontsize(9)
        return p, delta, effect
    return p, None, 'No'

print('Helper functions loaded.')


## Load Performance Data

In [ ]:
WT_THRESH, TC_THRESH, ET_THRESH = 0.91, 0.86, 0.85

unet_df     = load_unet_result('../../Results/Result/Vanilla_Unet/Unet_test_dice.csv')
nnunet_df   = load_nnunet_result('../../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json')
transbts_df = load_TransBTS_result('../../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json')

unet_nnunet_overlaps = get_overlaps(unet_df, transbts_df, nnunet_df)
performance_df = unet_df.copy()

print(f'Total cases:                    {len(unet_df)}')
print(f'Concordant poor (unet+nnunet):  {len(unet_nnunet_overlaps)}')
good_n = ((unet_df['WT dice'] >= WT_THRESH) & (unet_df['TC dice'] >= TC_THRESH) & (unet_df['ET dice'] >= ET_THRESH)).sum()
print(f'Concordant good:                {good_n}')


## Load First-Order Features for All Three Subregions

In [ ]:
print('Loading first-order features...')
wt_feat = load_firstorder('Tumor_WT')
tc_feat = load_firstorder('Tumor_TC')
et_feat = load_firstorder('Tumor_ET')

print(f'Tumor_WT firstorder shape: {wt_feat.shape}')
print(f'Tumor_TC firstorder shape: {tc_feat.shape}')
print(f'Tumor_ET firstorder shape: {et_feat.shape}')
print('\nSample columns (WT):', wt_feat.columns[:6].tolist())


In [ ]:
# Split each subregion into good / bad groups
wt_good, wt_bad = split_good_bad(wt_feat, performance_df, unet_nnunet_overlaps)
tc_good, tc_bad = split_good_bad(tc_feat, performance_df, unet_nnunet_overlaps)
et_good, et_bad = split_good_bad(et_feat, performance_df, unet_nnunet_overlaps)

wt_good_num, wt_bad_num = filter_numeric(wt_good), filter_numeric(wt_bad)
tc_good_num, tc_bad_num = filter_numeric(tc_good), filter_numeric(tc_bad)
et_good_num, et_bad_num = filter_numeric(et_good), filter_numeric(et_bad)

print(f'WT  — good: {len(wt_good_num)}, bad: {len(wt_bad_num)}')
print(f'TC  — good: {len(tc_good_num)}, bad: {len(tc_bad_num)}')
print(f'ET  — good: {len(et_good_num)}, bad: {len(et_bad_num)}')


## Plot: First-Order Features by Subregion and Modality

Layout: one figure per feature.  
Each figure has 4 subplots (one per MRI modality).  
Each subplot shows 3 groups (WT, TC, ET) × 2 boxes (Good, Bad).

In [ ]:
FEATURES = {
    'Mean':        'Intensity (Mean)',
    'Entropy':     'Entropy',
    'Kurtosis':    'Kurtosis',
    'Skewness':    'Skewness',
    'Uniformity':  'Uniformity',
    'Variance':    'Variance',
    'TotalEnergy': 'Energy',
}

MODALITIES = ['flair', 't2', 't1', 't1ce']
MOD_LABELS = ['FLAIR', 'T2', 'T1', 'T1CE']
SUBREGION_LABELS = ['WT', 'TC', 'ET']
COLORS = ['lightblue', 'lightcoral'] * 3

summary_data = {}

def col_name(feat, mod):
    """Column name as produced by load_firstorder(): original_firstorder_<Feat>_<mod>"""
    return f'original_firstorder_{feat}_{mod}'

with PdfPages('../../Results/final_figures/firstorder_subregion.pdf') as pdf:

    for feat_key, feat_title in FEATURES.items():
        summary_data[feat_key] = {}
        fig, axes = plt.subplots(1, 4, figsize=(18, 5))
        fig.suptitle(f'First-Order Feature: {feat_title}', fontsize=13)

        for ax, mod, mod_label in zip(axes, MODALITIES, MOD_LABELS):
            col = col_name(feat_key, mod)
            summary_data[feat_key][mod_label] = {}

            # [WT_good, WT_bad, TC_good, TC_bad, ET_good, ET_bad]
            groups = [
                (wt_good_num[col], wt_bad_num[col], 'WT'),
                (tc_good_num[col], tc_bad_num[col], 'TC'),
                (et_good_num[col], et_bad_num[col], 'ET'),
            ]

            box_data = [arr for g, b, _ in groups for arr in (g, b)]
            box = ax.boxplot(box_data, patch_artist=True, widths=0.5, showfliers=False)

            for patch, color in zip(box['boxes'], COLORS):
                patch.set_facecolor(color)
            for median in box['medians']:
                median.set_color('black')

            for i, (g, b, sr) in enumerate(groups):
                x_pos = 1 + 2 * i
                p, delta, effect = add_significance_indicator(ax, g, b, x_pos)
                summary_data[feat_key][mod_label][sr] = {
                    'median_good':  float(np.median(g)),
                    'iqr_good':     float(np.percentile(g, 75) - np.percentile(g, 25)),
                    'median_bad':   float(np.median(b)),
                    'iqr_bad':      float(np.percentile(b, 75) - np.percentile(b, 25)),
                    'p_value':      float(p),
                    'cliffs_delta': float(delta) if delta is not None else None,
                    'effect_size':  effect,
                }

            ax.set_xticks([1.5, 3.5, 5.5])
            ax.set_xticklabels(SUBREGION_LABELS)
            ax.set_title(mod_label)
            ax.set_ylabel('Feature Value' if ax == axes[0] else '')

            for j, x in enumerate(range(1, 7)):
                ax.text(
                    x,
                    ax.get_ylim()[0] - (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.1,
                    ['Good', 'Poor'][j % 2], ha='center', va='center', fontsize=8
                )
            for pos in [2.5, 4.5]:
                ax.axvline(x=pos, color='gray', linestyle='--', linewidth=1)

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.show()
        plt.close(fig)

print('PDF saved: ../../Results/final_figures/firstorder_subregion.pdf')


## Save JSON Summary

In [ ]:
with open('../../Results/Json_summary/summary_firstorder_subregion.json', 'w') as f:
    json.dump(summary_data, f, indent=4)

print('JSON saved: ../../Results/Json_summary/summary_firstorder_subregion.json')


## Print Summary Table

In [ ]:
rows = []
for feat, mods in summary_data.items():
    for mod, subregions in mods.items():
        for sr, stats in subregions.items():
            rows.append({
                'Feature':        feat,
                'Modality':       mod,
                'Subregion':      sr,
                'Median Good':    round(stats['median_good'], 4),
                'Median Bad':     round(stats['median_bad'], 4),
                'p-value':        f"{stats['p_value']:.3e}",
                "Cliff's Delta":  round(stats['cliffs_delta'], 3) if stats['cliffs_delta'] else 'n.s.',
                'Effect Size':    stats['effect_size'],
            })

results_df = pd.DataFrame(rows)

# Show significant results only
sig_df = results_df[results_df['Effect Size'] != 'No']
print(f'Total comparisons: {len(results_df)}')
print(f'Significant (p<0.05): {len(sig_df)}')
print(f'  Small effect: {(sig_df["Effect Size"] == "Small").sum()}')
print(f'  Medium effect: {(sig_df["Effect Size"] == "Medium").sum()}')
print(f'  Large effect: {(sig_df["Effect Size"] == "Large").sum()}')
print()
print('=== Significant results ===')
print(sig_df.to_string(index=False))


## Compare: WT-only (original) vs Per-Subregion (new)

Summarise which subregions show the strongest associations for each feature.

In [ ]:
print('=== Effect size breakdown by subregion ===')
pivot = sig_df.groupby(['Subregion', 'Effect Size']).size().unstack(fill_value=0)
print(pivot)

print('\n=== Effect size breakdown by modality ===')
pivot2 = sig_df.groupby(['Modality', 'Effect Size']).size().unstack(fill_value=0)
print(pivot2)

print('\n=== Features with Large effect in TC or ET (new findings vs WT-only) ===')
tc_et_large = sig_df[
    (sig_df['Subregion'].isin(['TC', 'ET'])) & 
    (sig_df['Effect Size'] == 'Large')
]
print(tc_et_large[['Feature', 'Modality', 'Subregion', 'Median Good', 'Median Bad', 'p-value', "Cliff's Delta"]].to_string(index=False))


## Single-Page Combined Figure (7 features × 4 modalities)

In [ ]:
def add_significance_text(ax, good, bad, x_pos, alpha=0.05):
    stat, p = mannwhitneyu(good, bad)
    if p < alpha:
        delta = cliffs_delta(good, bad)
        effect = categorize_effect_size(delta)
        ax.text(x_pos + 0.5, 1.02, f'* {effect}',
                transform=ax.get_xaxis_transform(),
                ha='center', va='bottom',
                fontsize=8, color='blue')
        return p, delta, effect
    return p, None, 'No'

# Features that need scaling for readability: {feat_key: (divisor, unit_label)}
SCALE = {
    'TotalEnergy': (1e9, '(×10⁹)'),
    'Variance':    (1e3, '(×10³)'),
}

def plot_combined(feat_keys, feat_titles_list, out_path):
    n_rows = len(feat_keys)
    n_cols = len(MODALITIES)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row, (feat_key, feat_title) in enumerate(zip(feat_keys, feat_titles_list)):
        divisor, unit = SCALE.get(feat_key, (1, ''))

        for col, (mod, mod_label) in enumerate(zip(MODALITIES, MOD_LABELS)):
            ax = axes[row, col]
            col_key = col_name(feat_key, mod)

            groups = [
                (wt_good_num[col_key] / divisor, wt_bad_num[col_key] / divisor, 'WT'),
                (tc_good_num[col_key] / divisor, tc_bad_num[col_key] / divisor, 'TC'),
                (et_good_num[col_key] / divisor, et_bad_num[col_key] / divisor, 'ET'),
            ]

            box_data = [arr for g, b, _ in groups for arr in (g, b)]
            bp = ax.boxplot(box_data, patch_artist=True, widths=0.5, showfliers=False)

            for patch, color in zip(bp['boxes'], COLORS):
                patch.set_facecolor(color)
            for median in bp['medians']:
                median.set_color('black')

            for i, (g, b, sr) in enumerate(groups):
                x_pos = 1 + 2 * i
                add_significance_text(ax, g, b, x_pos)

            ax.set_xticks([1.5, 3.5, 5.5])
            ax.set_xticklabels(['WT', 'TC', 'ET'], fontsize=9)
            for pos in [2.5, 4.5]:
                ax.axvline(x=pos, color='gray', linestyle='--', linewidth=0.8)

            if row == 0:
                ax.set_title(mod_label, fontsize=11, fontweight='bold', pad=22)

            if col == 0:
                ylabel = f'{feat_title} {unit}'.strip()
                ax.set_ylabel(ylabel, fontsize=11, labelpad=8)

    plt.tight_layout(h_pad=2.5)
    plt.savefig(out_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: {out_path}')

feat_list        = list(FEATURES.keys())
feat_titles_list = list(FEATURES.values())

plot_combined(
    feat_list[:4], feat_titles_list[:4],
    '../../Results/final_figures/firstorder_subregion_A.pdf'
)

plot_combined(
    feat_list[4:], feat_titles_list[4:],
    '../../Results/final_figures/firstorder_subregion_B.pdf'
)
